# 03 — Building an Information-Retrieval Dataset for Embedding Fine-Tuning

The previous notebooks transformed legal PDFs into validated articles. We now move
from **document engineering** to **machine-learning dataset engineering**.

The objective of this notebook is to create the data required to train and evaluate
an embedding model for Portuguese legal retrieval.

## Where we are in the pipeline

```text
Legal PDFs
   ↓
01 — Extraction
   ↓
Page-preserving text
   ↓
02 — Parsing and validation
   ↓
Validated legal articles
   ↓
03 — Dataset construction        ← this notebook
   ↓
Passages + queries + qrels + splits
   ↓
04 — Baseline retrieval evaluation
   ↓
05 — Hard-negative mining
   ↓
06 — Fine-tuning
```

## The retrieval problem

An embedding model receives a query such as:

```text
Quais são as regras sobre localização das operações?
```

and should rank a relevant legal passage above unrelated passages.

To evaluate this behaviour we need three fundamental objects:

```text
QUERY
"What are the rules about X?"
        │
        ▼
PASSAGE CORPUS
all searchable legal passages
        │
        ▼
QRELS
which passages are relevant to that query?
```

`qrels` means **query relevance judgements**. They are the ground truth used to
calculate retrieval metrics.

## Dataset design used here

Queries are generated at **article level**, while long articles may produce several
passages. Consequently, one query can have multiple relevant passages.

Two query families are kept separate:

- **reference queries** explicitly mention an article/document;
- **semantic queries** describe the legal topic without relying on the article number.

This distinction matters because retrieving `Artigo 6.º do CIVA` is easier than
understanding a natural-language information need.

## Learning objectives

After this notebook, you should understand:

- why legal articles may need to be divided into passages;
- why train/validation/test splitting must happen at article level;
- what queries, passages and qrels represent;
- the difference between reference and semantic retrieval;
- what positive pairs and triplets are;
- why passages from the same article must never become negatives;
- the difference between structural negatives and hard negatives;
- how deterministic generation supports reproducibility.

> This notebook creates the experimental dataset. It does **not** generate embeddings
> and it does **not** train the model yet.

## 1. Imports and paths

The input is `legal_corpus_ready.jsonl`, produced by notebook 02.

The output is written to `data/dataset_v2/`.

Several standard-library modules are important here:

- `hashlib` creates deterministic identifiers from content;
- `random` performs reproducible article splitting;
- `Counter` and `defaultdict` support dataset statistics and grouping;
- `re` is used for text and article-number processing;
- `json` reads and writes JSON Lines records.

At this stage, reproducibility becomes particularly important. A change in random
splitting can change evaluation results even when the model itself has not changed.

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import hashlib
import json
import random
import re

PROJECT_ROOT = Path("..")
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
DATASET_DIR = PROJECT_ROOT / "data" / "dataset"

DATASET_DIR.mkdir(parents=True, exist_ok=True)

CORPUS_PATH = PROCESSED_DIR / "legal_corpus_ready.jsonl"

print(f"Corpus : {CORPUS_PATH.resolve()}")
print(f"Dataset: {DATASET_DIR.resolve()}")

Corpus : C:\Users\user\Documents\GitHub\legal-embeddings-finetuning-pt\data\processed\legal_corpus_ready.jsonl
Dataset: C:\Users\user\Documents\GitHub\legal-embeddings-finetuning-pt\data\dataset_v2


## 2. Experimental configuration

These parameters define the dataset design and should be treated as part of the
experiment.

### Dataset split

```text
80%  training
10%  validation
10%  test
```

The **training set** is used to optimise the embedding model.

The **validation set** is used during model development, for example to compare
hyperparameters or select the best epoch.

The **test set** should remain untouched until the final evaluation.

### Passage size

`MAX_PASSAGE_CHARS` controls the approximate maximum passage length before an article
is divided into chunks. Character count is used here as a simple deterministic
pre-tokenisation limit.

`PASSAGE_OVERLAP_UNITS = 1` repeats one legal unit between neighbouring chunks. This
reduces the risk of losing context exactly at a chunk boundary.

### Query generation

Each article receives one explicit reference query and up to three semantic queries.

### Initial negatives

At this stage we create simple **structural negatives**. Later, notebook 05 will use
the baseline embedding model to find genuinely confusing passages — **hard negatives**.

`RANDOM_SEED = 42` ensures that the same corpus produces the same split and
deterministic sampling when the notebook is rerun.

In [2]:
RANDOM_SEED = 42

TRAIN_RATIO = 0.80
VALIDATION_RATIO = 0.10
TEST_RATIO = 0.10

MAX_PASSAGE_CHARS = 2200
PASSAGE_OVERLAP_UNITS = 1

# Reference queries explicitly mention the article/document.
REFERENCE_QUERIES_PER_ARTICLE = 1

# Semantic queries should dominate the training/evaluation dataset.
MAX_SEMANTIC_QUERIES_PER_ARTICLE = 3

# Number of structurally plausible negatives retained per training query.
NEGATIVES_PER_QUERY = 1
MAX_NEGATIVE_CANDIDATES = 30

random.seed(RANDOM_SEED)

assert abs(TRAIN_RATIO + VALIDATION_RATIO + TEST_RATIO - 1.0) < 1e-9
# These values are part of the experimental definition. Record them whenever
# results are reported so that later dataset versions remain comparable.


## 3. Load and validate the parsed corpus

Before constructing a machine-learning dataset, we verify that every input article
contains the metadata expected from notebook 02.

Required fields include:

- source document;
- legal block;
- article number and title;
- reconstructed text;
- source page range;
- stable article identifier;
- validation status.

Only records marked `ready` should reach this notebook.

This is a **pipeline contract**: notebook 03 assumes that notebook 02 has already
performed structural quality control. Checking the contract here prevents malformed
records from silently entering the training data.

In [3]:
def read_jsonl(path):
    records = []
    with path.open("r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(
                    f"Invalid JSON on line {line_number}: {exc}"
                )
    return records


articles = read_jsonl(CORPUS_PATH)

required_fields = {
    "document_id", "block_id", "article", "article_title", "text",
    "page_start", "page_end", "source_article_id", "validation_status"
}

errors = []

for article in articles:
    missing = required_fields - set(article)
    if missing:
        errors.append((article.get("source_article_id"), sorted(missing)))
    if article.get("validation_status") != "ready":
        errors.append((article.get("source_article_id"), "not_ready"))

print(f"Articles: {len(articles)}")
print("Documents:", Counter(a["document_id"] for a in articles))
print(f"Validation errors: {len(errors)}")

if errors:
    for error in errors[:50]:
        print(error)

Articles: 1329
Documents: Counter({'oe2026': 249, 'cirs': 191, 'circ': 167, 'cimi': 153, 'rgit': 123, 'ebf': 122, 'lgt': 118, 'civa': 100, 'cis': 73, 'riti': 33})
Validation errors: 0


## 4. Text and metadata helpers

These helper functions standardise the representation that will later be shown to the
embedding model.

A passage will not contain only the body of the article. It can also include useful
context such as:

```text
Documento: CIVA
Estrutura: Título I | Capítulo II
Artigo 6.º — Localização das operações
<body>
```

This metadata can help distinguish provisions that use similar vocabulary in different
legal contexts.

`document_label()` converts technical IDs such as `civa` into readable labels.

`hierarchy_label()` reconstructs the hierarchy captured by the parser.

`article_heading()` creates a consistent human-readable article heading.

The objective is to enrich the passage with **source context**, not to invent new legal
content.

In [4]:
def clean_text(text):
    text = (text or "").strip()
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    return text


def document_label(document_id):
    labels = {
        "civa": "CIVA",
        "cirs": "CIRS",
        "circ": "CIRC",
        "cimi": "CIMI",
        "cis": "Código do Imposto do Selo",
        "ebf": "Estatuto dos Benefícios Fiscais",
        "lgt": "Lei Geral Tributária",
        "rgit": "RGIT",
        "riti": "RITI",
        "oe2026": "Orçamento do Estado para 2026",
    }
    return labels.get(document_id, document_id.upper())


def hierarchy_label(article):
    parts = []
    for level in ["part", "book", "title", "chapter", "section", "subsection"]:
        value = article.get(level)
        heading = article.get(f"{level}_title")
        if value:
            label = f"{level.capitalize()} {value}"
            if heading:
                label += f" — {heading}"
            parts.append(label)
    return " | ".join(parts)


def article_heading(article):
    heading = f"Artigo {article['article']}"
    if article.get("article_title"):
        heading += f" — {article['article_title']}"
    return heading

## 5. Build article passages

An article is the natural legal unit, but some articles are too long to be encoded
comfortably as one retrieval passage.

We therefore create one or more **passages** per article.

### Preferred splitting strategy

The function first tries to split the article at numbered legal units:

```text
1 - ...
2 - ...
3 - ...
```

This is preferable to cutting every N characters because it respects the internal
structure of the provision.

If the article is short enough:

```text
1 article → 1 passage
```

If it is long:

```text
1 article
   ├── passage 1
   ├── passage 2
   └── passage 3
```

Every passage repeats the document/article prefix so it remains interpretable when
retrieved independently.

### Overlap

One body unit can be repeated between adjacent chunks:

```text
chunk 1: paragraphs 1, 2, 3
chunk 2: paragraphs 3, 4, 5
```

Overlap preserves local context around chunk boundaries.

### Fallback

If no numbered units can be identified, a long uninterrupted body is divided by
character length. This is a fallback rather than the preferred legal segmentation.

In [5]:
def split_body_units(text):
    text = clean_text(text)

    units = [
        u.strip()
        for u in re.split(r"\n(?=\d+\s*[-–—]\s+)", text)
        if u.strip()
    ]

    if len(units) <= 1:
        units = [u.strip() for u in text.split("\n") if u.strip()]

    return units


def passage_prefix(article):
    lines = [f"Documento: {document_label(article['document_id'])}"]

    hierarchy = hierarchy_label(article)
    if hierarchy:
        lines.append(f"Estrutura: {hierarchy}")

    lines.append(article_heading(article))
    return "\n".join(lines)


def chunk_article(article, max_chars=MAX_PASSAGE_CHARS, overlap_units=PASSAGE_OVERLAP_UNITS):
    prefix = passage_prefix(article)
    body = clean_text(article.get("text", ""))
    full = f"{prefix}\n{body}".strip()

    if len(full) <= max_chars:
        return [full]

    units = split_body_units(body)

    # Fallback for a single unbroken unit.
    if len(units) == 1 and len(full) > max_chars:
        body_limit = max(500, max_chars - len(prefix) - 1)
        return [
            f"{prefix}\n{body[i:i + body_limit]}".strip()
            for i in range(0, len(body), body_limit)
        ]

    chunks = []
    current = []

    for unit in units:
        candidate = f"{prefix}\n" + "\n".join(current + [unit])

        if current and len(candidate) > max_chars:
            chunks.append(f"{prefix}\n" + "\n".join(current))

            current = (
                current[-overlap_units:]
                if overlap_units > 0
                else []
            )

        current.append(unit)

    if current:
        chunks.append(f"{prefix}\n" + "\n".join(current))

    return [c.strip() for c in chunks if c.strip()]

In [6]:
passages = []

for article in articles:
    chunks = chunk_article(article)

    for index, text in enumerate(chunks, start=1):
        passage_id = article["source_article_id"]
        if len(chunks) > 1:
            passage_id += f"__chunk_{index:02d}"

        passages.append({
            "passage_id": passage_id,
            "source_article_id": article["source_article_id"],
            "document_id": article["document_id"],
            "block_id": article["block_id"],
            "article": article["article"],
            "article_title": article.get("article_title"),
            "chunk_index": index,
            "chunk_count": len(chunks),
            "page_start": article["page_start"],
            "page_end": article["page_end"],
            "text": text,
            "character_count": len(text),
        })

passages_by_article = defaultdict(list)
for passage in passages:
    passages_by_article[passage["source_article_id"]].append(passage)

print(f"Articles : {len(articles)}")
print(f"Passages : {len(passages)}")
print(f"Chunked articles: {sum(len(v) > 1 for v in passages_by_article.values())}")
print(f"Maximum passage length: {max(p['character_count'] for p in passages)}")
# `passages_by_article` is important because one article may map to several
# passages. Evaluation can therefore treat every chunk of the target article
# as relevant instead of arbitrarily choosing only one.


Articles : 1329
Passages : 2088
Chunked articles: 284
Maximum passage length: 11650


## 6. Article-level train / validation / test split

This is one of the most important methodological decisions in the notebook.

The split is performed at **article level**, not passage level and not query level.

Why?

Imagine Article 10 produces three chunks:

```text
Article 10
├── chunk 1
├── chunk 2
└── chunk 3
```

If chunk 1 were placed in training and chunk 2 in test, the model would already have
seen text from the same legal provision during training. Evaluation would therefore
overestimate generalisation.

Instead:

```text
Article 10 → TRAIN
    ├── chunk 1 → TRAIN
    ├── chunk 2 → TRAIN
    └── every query for Article 10 → TRAIN
```

The same rule applies to validation and test.

The split is performed independently inside each document so that the major legal
sources remain represented across subsets where their size permits it.

The assertions at the end explicitly verify that no article identifier occurs in more
than one split. This is a direct test for **data leakage**.

In [7]:
articles_by_document = defaultdict(list)

for article in articles:
    articles_by_document[article["document_id"]].append(
        article["source_article_id"]
    )

article_split = {}
rng = random.Random(RANDOM_SEED)

for document_id, ids in articles_by_document.items():
    ids = sorted(set(ids))
    rng.shuffle(ids)

    n = len(ids)
    n_train = int(n * TRAIN_RATIO)
    n_validation = int(n * VALIDATION_RATIO)

    if n >= 10:
        n_validation = max(1, n_validation)

    n_test = n - n_train - n_validation

    if n >= 10 and n_test == 0:
        n_test = 1
        n_train = max(1, n_train - 1)

    for article_id in ids[:n_train]:
        article_split[article_id] = "train"

    for article_id in ids[n_train:n_train + n_validation]:
        article_split[article_id] = "validation"

    for article_id in ids[n_train + n_validation:]:
        article_split[article_id] = "test"

for passage in passages:
    passage["split"] = article_split[passage["source_article_id"]]

print(Counter(article_split.values()))

split_sets = {
    split: {k for k, v in article_split.items() if v == split}
    for split in ["train", "validation", "test"]
}

assert split_sets["train"].isdisjoint(split_sets["validation"])
assert split_sets["train"].isdisjoint(split_sets["test"])
assert split_sets["validation"].isdisjoint(split_sets["test"])

print("No article-level leakage between splits.")
# The split is propagated from articles to passages only after the article-level
# assignment has been fixed. This is what prevents sibling chunks leaking across
# train, validation and test.


Counter({'train': 1059, 'test': 141, 'validation': 129})
No article-level leakage between splits.


## 7. Query generation strategy

A retrieval dataset needs realistic information needs, not only legal passages.

This notebook creates queries **once per article**.

### 7.1 Reference queries

A reference query explicitly identifies the target provision:

```text
O que estabelece o artigo 6.º do CIVA?
```

This tests navigational retrieval. The model can exploit highly discriminative tokens
such as the document name and article number.

### 7.2 Semantic queries

A semantic query asks about the legal concept:

```text
Quais são as regras sobre localização das operações?
Como é regulada a matéria de localização das operações no CIVA?
```

The article number is absent. Retrieval therefore depends more strongly on semantic
similarity.

### Why derive them from article titles?

The title is a deterministic semantic signal already present in the source corpus.
Using it avoids asking a generative model to invent questions at this stage.

This gives us a reproducible baseline, but there is an important limitation:
these questions are **synthetic templates**, not genuine questions collected from
people. Later work can augment them with LLM-generated questions and human review.

Articles with generic titles such as `Objeto` or `Disposições gerais` are deliberately
treated cautiously because such titles produce weak semantic queries.

In [8]:
GENERIC_TITLES = {
    "", "objeto", "objecto", "âmbito", "definições", "definicoes",
    "disposições gerais", "disposicoes gerais"
}


def normalize_topic(title):
    title = clean_text(title)
    title = re.sub(r"[.;:]+$", "", title).strip()
    return title


def usable_title(title):
    if not title:
        return False

    topic = normalize_topic(title)
    if len(topic) < 4:
        return False

    return topic.lower() not in GENERIC_TITLES


def generate_article_queries(article):
    document = document_label(article["document_id"])
    article_number = article["article"]
    title = article.get("article_title")

    queries = []

    # One explicit reference query.
    queries.append({
        "query_type": "reference",
        "query": f"O que estabelece o artigo {article_number} do {document}?"
    })

    if usable_title(title):
        topic = normalize_topic(title)

        semantic = [
            f"Quais são as regras sobre {topic}?",
            f"Como é regulada a matéria de {topic} no {document}?",
            f"O que prevê o {document} relativamente a {topic}?",
        ]

        for query in semantic[:MAX_SEMANTIC_QUERIES_PER_ARTICLE]:
            queries.append({
                "query_type": "semantic",
                "query": query
            })

    # Stable de-duplication.
    seen = set()
    output = []

    for item in queries:
        key = item["query"].strip().lower()
        if key not in seen:
            seen.add(key)
            output.append(item)

    return output

## 8. Create article-level retrieval queries

Each query stores **all passages belonging to the target article** as relevant
passages. This is the key change required for correct retrieval evaluation.

In [9]:
article_by_id = {
    article["source_article_id"]: article
    for article in articles
}

retrieval_queries = []

for article in articles:
    article_id = article["source_article_id"]
    relevant_passages = [
        p["passage_id"]
        for p in passages_by_article[article_id]
    ]

    for query_index, item in enumerate(generate_article_queries(article), start=1):
        key = (
            article_id
            + "|"
            + item["query_type"]
            + "|"
            + item["query"]
        )

        query_id = hashlib.sha256(
            key.encode("utf-8")
        ).hexdigest()[:16]

        retrieval_queries.append({
            "query_id": query_id,
            "split": article_split[article_id],
            "query_type": item["query_type"],
            "query": item["query"],
            "document_id": article["document_id"],
            "article": article["article"],
            "article_title": article.get("article_title"),
            "source_article_id": article_id,
            "relevant_passage_ids": relevant_passages,
            "relevant_passage_count": len(relevant_passages),
            "query_index": query_index,
        })

print(f"Queries: {len(retrieval_queries)}")
print("By type :", Counter(q["query_type"] for q in retrieval_queries))
print("By split:", Counter(q["split"] for q in retrieval_queries))

Queries: 5301
By type : Counter({'semantic': 3972, 'reference': 1329})
By split: Counter({'train': 4224, 'test': 561, 'validation': 516})


## 9. Query diagnostics

Before using generated queries for evaluation, inspect their distribution.

For each split we report:

- total queries;
- query types;
- number of distinct source articles.

This helps detect problems such as one split receiving unexpectedly few semantic
queries or a large imbalance introduced by document structure.

Dataset diagnostics should happen **before** model evaluation. Otherwise a surprising
metric may actually be caused by an unexpected dataset composition.

In [10]:
for split in ["train", "validation", "test"]:
    subset = [q for q in retrieval_queries if q["split"] == split]

    print(f"\n{split.upper()}")
    print(" total   :", len(subset))
    print(" types   :", Counter(q["query_type"] for q in subset))
    print(
        " articles:",
        len(set(q["source_article_id"] for q in subset))
    )


TRAIN
 total   : 4224
 types   : Counter({'semantic': 3165, 'reference': 1059})
 articles: 1059

VALIDATION
 total   : 516
 types   : Counter({'semantic': 387, 'reference': 129})
 articles: 129

TEST
 total   : 561
 types   : Counter({'semantic': 420, 'reference': 141})
 articles: 141


## 10. Inspect semantic queries manually

Automatic generation should always be followed by human inspection.

A sample of semantic queries is printed so we can ask:

- Does the question make sense in Portuguese?
- Is it specific enough to identify the intended legal topic?
- Is the wording too mechanical?
- Is the article title itself truncated or malformed?
- Could several unrelated articles plausibly answer the same question?

This inspection is especially important because later hard-negative mining can expose
weak queries that are inherently ambiguous.

In [11]:
semantic_queries = [
    q for q in retrieval_queries
    if q["query_type"] == "semantic"
]

sample_rng = random.Random(RANDOM_SEED)

for query in sample_rng.sample(
    semantic_queries,
    min(30, len(semantic_queries))
):
    print(
        f"{query['document_id'].upper():8} | "
        f"Art. {query['article']:8} | "
        f"{query['query']}"
    )

LGT      | Art. 63.º-D   | Quais são as regras sobre Países, territórios ou regiões com um regime fiscal claramente mais favorável?
CIMI     | Art. 140.º    | Quais são as regras sobre Regime de salvaguarda de prédios urbanos?
CIMI     | Art. 34.º     | Quais são as regras sobre Operações de avaliação?
RGIT     | Art. 103.º    | Como é regulada a matéria de Fraude no RGIT?
CIRS     | Art. 45.º     | Como é regulada a matéria de Valor de aquisição a título gratuito no CIRS?
CIRS     | Art. 12.º-B   | Como é regulada a matéria de Isenção de rendimentos das categorias A e B no CIRS?
CIRC     | Art. 127.º    | O que prevê o CIRC relativamente a Deveres de cooperação dos organismos oficiais e de outras entidades?
CIRC     | Art. 32.º     | Como é regulada a matéria de Projetos de desenvolvimento no CIRC?
RGIT     | Art. 96.º     | Como é regulada a matéria de Introdução fraudulenta no consumo no RGIT?
CIMI     | Art. 135.º-D  | O que prevê o CIMI relativamente a Sujeitos passivos casados ou

## 11. Build evaluation qrels

A **qrel** is a relevance judgement connecting a query to a relevant passage.

For example:

```text
query_123 → passage_CIVA_article_6 → relevance = 1
```

If an article has three chunks, all three are considered relevant for its article-level
query:

```text
query_123
   ├── chunk_01 → relevant
   ├── chunk_02 → relevant
   └── chunk_03 → relevant
```

This is why evaluation metrics such as Recall@K are useful: the system may need to
retrieve several relevant passages.

The relevance labels here are binary:

```text
1 = relevant
```

No graded distinction such as `highly relevant` versus `partially relevant` is made in
this version.

In [12]:
qrels = []

for query in retrieval_queries:
    for passage_id in query["relevant_passage_ids"]:
        qrels.append({
            "query_id": query["query_id"],
            "passage_id": passage_id,
            "relevance": 1,
            "split": query["split"],
            "query_type": query["query_type"],
            "source_article_id": query["source_article_id"],
        })

print(f"Qrels: {len(qrels)}")
# Qrels are the ground truth for ranking evaluation. They are kept separate from
# queries so that the same query set can be evaluated with different models.


Qrels: 8337


## 12. Negative candidate pools

Training a retrieval model requires examples of both relevant and non-relevant text.

For a query about Article 20:

```text
positive → passage from Article 20
negative → passage from another article
```

The function below builds **structural negative candidates** using simple legal
proximity heuristics.

Preference is given to:

1. passages from the same document;
2. articles with nearby numbers;
3. titled provisions;
4. other passages in the same split.

A nearby article in the same code is usually more informative than an obviously
unrelated passage from another document.

### Critical safety rule

No passage from the target article can be a negative — including another chunk of the
same long article.

### Structural negatives vs. hard negatives

These are not yet true hard negatives.

A **hard negative** is a non-relevant passage that the current embedding model ranks
very highly because it looks semantically similar to the query.

Those will be mined after baseline evaluation in notebook 05.

In [13]:
passages_by_split = defaultdict(list)

for passage in passages:
    passages_by_split[passage["split"]].append(passage)


def numeric_article_value(article):
    match = re.match(r"(\d+)", article or "")
    return int(match.group(1)) if match else 10**9


def negative_candidates(query, max_candidates=MAX_NEGATIVE_CANDIDATES):
    positive_article = article_by_id[query["source_article_id"]]
    positive_number = numeric_article_value(positive_article["article"])

    candidates = [
        p
        for p in passages_by_split[query["split"]]
        if p["source_article_id"] != query["source_article_id"]
    ]

    def rank(passage):
        same_document = passage["document_id"] == query["document_id"]
        distance = abs(
            numeric_article_value(passage["article"])
            - positive_number
        )

        # A titled article in the same document is usually a more plausible
        # structural negative than a random passage from another code.
        has_title = bool(passage.get("article_title"))

        return (
            0 if same_document else 1,
            distance,
            0 if has_title else 1,
            passage["passage_id"],
        )

    candidates.sort(key=rank)
    return candidates[:max_candidates]

## 13. Build training triplets without sibling-chunk conflicts

A triplet has the form:

```text
(anchor, positive, negative)

anchor   = query
positive = relevant legal passage
negative = non-relevant legal passage
```

Example:

```text
Query:
"Quais são as regras sobre localização das operações?"

Positive:
CIVA — Artigo 6.º ...

Negative:
CIVA — another plausible but non-relevant article ...
```

Triplet-based training encourages the embedding space to satisfy:

```text
similarity(query, positive)
        >
similarity(query, negative)
```

For long articles, one positive chunk is selected deterministically. Other chunks from
that same article remain relevant in qrels and are excluded from the negative pool.

`stable_index()` uses SHA-256 rather than ordinary random sampling. The same query ID
therefore selects the same positive/negative candidate every time the dataset is
rebuilt, which makes the experiment auditable.

In [14]:
def stable_index(text, modulo):
    if modulo <= 0:
        return 0

    digest = hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()

    return int(digest[:12], 16) % modulo


triplets = []

for query in retrieval_queries:
    positive_passages = passages_by_article[
        query["source_article_id"]
    ]

    positive_index = stable_index(
        query["query_id"] + "|positive",
        len(positive_passages)
    )

    positive = positive_passages[positive_index]

    candidates = negative_candidates(query)

    if not candidates:
        continue

    for negative_number in range(NEGATIVES_PER_QUERY):
        negative_index = stable_index(
            query["query_id"] + f"|negative|{negative_number}",
            len(candidates)
        )

        negative = candidates[negative_index]

        triplet_id = hashlib.sha256(
            (
                query["query_id"]
                + "|"
                + positive["passage_id"]
                + "|"
                + negative["passage_id"]
            ).encode("utf-8")
        ).hexdigest()[:16]

        triplets.append({
            "triplet_id": triplet_id,
            "query_id": query["query_id"],
            "split": query["split"],
            "query_type": query["query_type"],
            "query": query["query"],
            "positive": positive["text"],
            "negative": negative["text"],
            "positive_passage_id": positive["passage_id"],
            "negative_passage_id": negative["passage_id"],
            "source_article_id": query["source_article_id"],
            "document_id": query["document_id"],
            "article": query["article"],
        })

print(f"Triplets: {len(triplets)}")
print(Counter(t["query_type"] for t in triplets))
print(Counter(t["split"] for t in triplets))

Triplets: 5301
Counter({'semantic': 3972, 'reference': 1329})
Counter({'train': 4224, 'test': 561, 'validation': 516})


## 14. Build positive pairs

Some embedding objectives use only positive pairs rather than explicit triplets:

```text
(query, positive passage)
```

The notebook exports both representations so later experiments can compare different
training objectives without rebuilding the entire dataset.

For example:

- triplets can support `TripletLoss`;
- positive pairs can support contrastive approaches such as
  `MultipleNegativesRankingLoss`.

Keeping both formats increases experimental flexibility.

In [15]:
pairs = [
    {
        "query_id": t["query_id"],
        "split": t["split"],
        "query_type": t["query_type"],
        "query": t["query"],
        "positive": t["positive"],
        "positive_passage_id": t["positive_passage_id"],
        "source_article_id": t["source_article_id"],
        "document_id": t["document_id"],
        "article": t["article"],
    }
    for t in triplets
]

print(f"Pairs: {len(pairs)}")

Pairs: 5301


## 15. Dataset integrity checks

Before exporting the dataset, we test several invariants.

The checks verify that:

- query IDs are unique;
- every query has at least one relevant passage;
- every qrel points to an existing passage;
- positive and negative IDs are different;
- a sibling chunk from the same article is never used as a negative.

These are not cosmetic checks. A false negative from the same source article would
teach the model contradictory information:

```text
"This passage is relevant"
and
"This other passage from the same target article is irrelevant"
```

Such label noise can directly damage fine-tuning.

This is another example of why dataset engineering is often as important as the model
architecture itself.

In [16]:
quality_errors = []

query_ids = [q["query_id"] for q in retrieval_queries]
if len(query_ids) != len(set(query_ids)):
    quality_errors.append("duplicate_query_ids")

passage_ids = {p["passage_id"] for p in passages}

for query in retrieval_queries:
    if not query["relevant_passage_ids"]:
        quality_errors.append(
            f"no_relevant_passage:{query['query_id']}"
        )

    for pid in query["relevant_passage_ids"]:
        if pid not in passage_ids:
            quality_errors.append(
                f"missing_qrel_passage:{query['query_id']}:{pid}"
            )

for triplet in triplets:
    if triplet["positive_passage_id"] == triplet["negative_passage_id"]:
        quality_errors.append(
            f"same_positive_negative:{triplet['triplet_id']}"
        )

    negative = next(
        p for p in passages
        if p["passage_id"] == triplet["negative_passage_id"]
    )

    if negative["source_article_id"] == triplet["source_article_id"]:
        quality_errors.append(
            f"sibling_chunk_used_as_negative:{triplet['triplet_id']}"
        )

print(f"Quality errors: {len(quality_errors)}")

for error in quality_errors[:50]:
    print(error)

Quality errors: 0


## 16. Reference versus semantic balance

We report the proportion of semantic queries in each split.

A model can obtain deceptively strong retrieval results if most evaluation questions
contain exact article references. Semantic queries provide a more demanding test of
whether the embedding space captures legal meaning.

For this project, the two query types should therefore be reported separately during
baseline and final evaluation.

In [17]:
query_type_stats = {}

for split in ["train", "validation", "test"]:
    subset = [
        q for q in retrieval_queries
        if q["split"] == split
    ]

    counts = Counter(
        q["query_type"]
        for q in subset
    )

    query_type_stats[split] = dict(counts)

    total = len(subset)
    semantic = counts.get("semantic", 0)

    print(
        f"{split:10} | total={total:5} | "
        f"semantic={semantic:5} "
        f"({semantic / total:.1%})"
        if total else split
    )

train      | total= 4224 | semantic= 3165 (74.9%)
validation | total=  516 | semantic=  387 (75.0%)
test       | total=  561 | semantic=  420 (74.9%)


## 17. Representative retrieval examples

Before exporting thousands of records, inspect a concrete query and its relevant
passage for each legal document.

This creates a human-readable sanity check of the complete relationship:

```text
source article
     ↓
generated semantic query
     ↓
relevant passage ID
     ↓
actual passage text
```

If this relationship looks wrong here, downstream retrieval metrics will not be
meaningful.

In [18]:
for document_id in sorted(set(a["document_id"] for a in articles)):
    candidates = [
        q for q in retrieval_queries
        if (
            q["document_id"] == document_id
            and q["query_type"] == "semantic"
        )
    ]

    if not candidates:
        continue

    query = candidates[0]

    print("\n" + "=" * 90)
    print(f"{document_id.upper()} | {query['split']}")
    print("QUERY:")
    print(query["query"])
    print("RELEVANT PASSAGES:")
    print(query["relevant_passage_ids"])
    print("FIRST RELEVANT PASSAGE:")
    print(
        passages_by_article[
            query["source_article_id"]
        ][0]["text"][:1200]
    )


CIMI | train
QUERY:
Quais são as regras sobre Incidência?
RELEVANT PASSAGES:
['cimi_block_02_art_1_o']
FIRST RELEVANT PASSAGE:
Documento: CIMI
Artigo 1.º — Incidência
1 - O imposto municipal sobre imóveis (IMI) incide sobre o valor patrimonial tributário dos prédios
rústicos e urbanos situados no território português, constituindo receita dos municípios onde os
mesmos se localizam. (Redação da Lei n.o 42/2016, de 28 de dezembro; anterior proémio do artigo)
2 - O adicional ao imposto municipal sobre imóveis, deduzido dos encargos de cobrança e da
previsão de deduções à coleta de imposto sobre o rendimento das pessoas singulares (IRS) e
de imposto sobre o rendimento das pessoas coletivas (IRC), constitui receita do Fundo de
Estabilização Financeira da Segurança Social. (Redação da Lei n.o 114/2017, de 29 de dezembro)

CIRC | train
QUERY:
Quais são as regras sobre Pressuposto do imposto?
RELEVANT PASSAGES:
['circ_block_01_art_1_o']
FIRST RELEVANT PASSAGE:
Documento: CIRC
Artigo 1.º — Pre

## 18. Dataset statistics

The statistics object creates a compact description of the experimental dataset.

It records global counts and per-document counts for:

- articles;
- passages;
- queries;
- semantic queries;
- qrels;
- pairs;
- triplets;
- split membership.

These values are useful for reports, experiment tracking and later comparisons between
dataset versions.

A model result without a clear description of the dataset on which it was obtained is
difficult to reproduce.

In [19]:
dataset_statistics = {
    "dataset_version": "2.0",
    "articles": len(articles),
    "passages": len(passages),
    "queries": len(retrieval_queries),
    "qrels": len(qrels),
    "pairs": len(pairs),
    "triplets": len(triplets),
    "query_types": dict(Counter(q["query_type"] for q in retrieval_queries)),
    "split_articles": dict(Counter(article_split.values())),
    "split_queries": dict(Counter(q["split"] for q in retrieval_queries)),
    "split_triplets": dict(Counter(t["split"] for t in triplets)),
    "documents": {},
}

for document_id in sorted(set(a["document_id"] for a in articles)):
    dataset_statistics["documents"][document_id] = {
        "articles": sum(a["document_id"] == document_id for a in articles),
        "passages": sum(p["document_id"] == document_id for p in passages),
        "queries": sum(q["document_id"] == document_id for q in retrieval_queries),
        "semantic_queries": sum(
            q["document_id"] == document_id and q["query_type"] == "semantic"
            for q in retrieval_queries
        ),
    }

dataset_statistics

{'dataset_version': '2.0',
 'articles': 1329,
 'passages': 2088,
 'queries': 5301,
 'qrels': 8337,
 'pairs': 5301,
 'triplets': 5301,
 'query_types': {'reference': 1329, 'semantic': 3972},
 'split_articles': {'train': 1059, 'validation': 129, 'test': 141},
 'split_queries': {'train': 4224, 'validation': 516, 'test': 561},
 'split_triplets': {'train': 4224, 'validation': 516, 'test': 561},
 'documents': {'cimi': {'articles': 153,
   'passages': 177,
   'queries': 612,
   'semantic_queries': 459},
  'circ': {'articles': 167,
   'passages': 360,
   'queries': 668,
   'semantic_queries': 501},
  'cirs': {'articles': 191,
   'passages': 334,
   'queries': 764,
   'semantic_queries': 573},
  'cis': {'articles': 73,
   'passages': 101,
   'queries': 292,
   'semantic_queries': 219},
  'civa': {'articles': 100,
   'passages': 204,
   'queries': 400,
   'semantic_queries': 300},
  'ebf': {'articles': 122,
   'passages': 258,
   'queries': 485,
   'semantic_queries': 363},
  'lgt': {'articles': 

## 19. JSONL export helper

The dataset is exported using **JSON Lines (JSONL)**.

Unlike a normal JSON array, JSONL stores one independent JSON object per line:

```text
{"query_id": "...", "query": "..."}
{"query_id": "...", "query": "..."}
{"query_id": "...", "query": "..."}
```

This format is common in machine-learning pipelines because it is easy to stream,
inspect and process record by record.

`ensure_ascii=False` preserves Portuguese characters directly.

In [20]:
def write_jsonl(path, records):
    with path.open("w", encoding="utf-8") as f:
        for record in records:
            f.write(
                json.dumps(record, ensure_ascii=False)
                + "\n"
            )
# One JSON object is written per line. This makes the exported files easy to
# stream and inspect without loading the entire dataset into memory.


## 20. Export the passage corpus

`passages.jsonl` is the searchable corpus.

Every retrieval experiment in the next stages will embed these passage texts and rank
them against query embeddings.

Each passage also retains metadata such as document, article, chunk index and source
pages, allowing retrieved vectors to be traced back to legal provisions.

In [21]:
PASSAGES_PATH = DATASET_DIR / "passages.jsonl"
write_jsonl(PASSAGES_PATH, passages)

print(PASSAGES_PATH.resolve())

C:\Users\user\Documents\GitHub\legal-embeddings-finetuning-pt\data\dataset_v2\passages.jsonl


## 21. Export queries and qrels by split

Queries and relevance judgements are exported separately for train, validation and
test.

This separation makes the intended use explicit:

```text
train       → model optimisation
validation  → development and checkpoint selection
test        → final unbiased evaluation
```

The test files should not be used to choose hyperparameters or decide which checkpoint
is best.

In [22]:
for split in ["train", "validation", "test"]:
    split_queries = [
        q for q in retrieval_queries
        if q["split"] == split
    ]

    split_qrels = [
        qrel for qrel in qrels
        if qrel["split"] == split
    ]

    write_jsonl(
        DATASET_DIR / f"{split}_queries.jsonl",
        split_queries
    )

    write_jsonl(
        DATASET_DIR / f"{split}_qrels.jsonl",
        split_qrels
    )

    print(
        f"{split:10} | "
        f"queries={len(split_queries):5} | "
        f"qrels={len(split_qrels):5}"
    )

train      | queries= 4224 | qrels= 6752
validation | queries=  516 | qrels=  632
test       | queries=  561 | qrels=  953


## 22. Export positive pairs and triplets

Both training representations are exported for every split.

Although later fine-tuning may use only the training triplets, exporting validation
and test representations can be useful for diagnostics and alternative experiments.

The authoritative retrieval evaluation, however, should use **queries + qrels +
passage corpus**, because that evaluates ranking against the full candidate corpus
rather than against one preselected negative.

In [23]:
for split in ["train", "validation", "test"]:
    split_pairs = [p for p in pairs if p["split"] == split]
    split_triplets = [t for t in triplets if t["split"] == split]

    write_jsonl(
        DATASET_DIR / f"{split}_pairs.jsonl",
        split_pairs
    )

    write_jsonl(
        DATASET_DIR / f"{split}_triplets.jsonl",
        split_triplets
    )

    print(
        f"{split:10} | "
        f"pairs={len(split_pairs):5} | "
        f"triplets={len(split_triplets):5}"
    )

train      | pairs= 4224 | triplets= 4224
validation | pairs=  516 | triplets=  516
test       | pairs=  561 | triplets=  561


## 23. Export the article split manifest

The split manifest records exactly which article belongs to train, validation or test.

This small file is important for reproducibility. If the dataset is rebuilt later, we
can inspect whether the article assignment changed.

It also provides a simple way to audit the central no-leakage rule.

In [24]:
SPLIT_MANIFEST_PATH = DATASET_DIR / "article_split_manifest.json"

with SPLIT_MANIFEST_PATH.open("w", encoding="utf-8") as f:
    json.dump(
        article_split,
        f,
        ensure_ascii=False,
        indent=2
    )

print(SPLIT_MANIFEST_PATH.resolve())

C:\Users\user\Documents\GitHub\legal-embeddings-finetuning-pt\data\dataset_v2\article_split_manifest.json


## 24. Export the dataset report

The report stores both configuration and resulting statistics.

This means the dataset directory documents not only **what records were produced** but
also **how they were produced**.

Examples of recorded parameters include:

- split ratios;
- passage-size threshold;
- overlap;
- number of semantic queries;
- number of negatives;
- random seed;
- quality errors.

This is lightweight experiment metadata and becomes valuable when several dataset
versions exist.

In [25]:
REPORT_PATH = DATASET_DIR / "_dataset_report.json"

report = {
    "dataset_version": "2.0",
    "random_seed": RANDOM_SEED,
    "source_corpus": str(CORPUS_PATH),
    "configuration": {
        "train_ratio": TRAIN_RATIO,
        "validation_ratio": VALIDATION_RATIO,
        "test_ratio": TEST_RATIO,
        "max_passage_chars": MAX_PASSAGE_CHARS,
        "passage_overlap_units": PASSAGE_OVERLAP_UNITS,
        "reference_queries_per_article": REFERENCE_QUERIES_PER_ARTICLE,
        "max_semantic_queries_per_article": MAX_SEMANTIC_QUERIES_PER_ARTICLE,
        "negatives_per_query": NEGATIVES_PER_QUERY,
        "max_negative_candidates": MAX_NEGATIVE_CANDIDATES,
    },
    "statistics": dataset_statistics,
    "query_type_stats": query_type_stats,
    "quality_errors": quality_errors,
}

with REPORT_PATH.open("w", encoding="utf-8") as f:
    json.dump(
        report,
        f,
        ensure_ascii=False,
        indent=2
    )

print(REPORT_PATH.resolve())

C:\Users\user\Documents\GitHub\legal-embeddings-finetuning-pt\data\dataset_v2\_dataset_report.json


## 25. Final output structure

At the end of this notebook we have transformed validated legislation into a complete
information-retrieval dataset:

```text
data/
└── dataset_v2/
    ├── passages.jsonl
    ├── article_split_manifest.json
    │
    ├── train_queries.jsonl
    ├── validation_queries.jsonl
    ├── test_queries.jsonl
    │
    ├── train_qrels.jsonl
    ├── validation_qrels.jsonl
    ├── test_qrels.jsonl
    │
    ├── train_pairs.jsonl
    ├── validation_pairs.jsonl
    ├── test_pairs.jsonl
    │
    ├── train_triplets.jsonl
    ├── validation_triplets.jsonl
    ├── test_triplets.jsonl
    │
    └── _dataset_report.json
```

## What each object means

```text
PASSAGE
a searchable piece of legislation

QUERY
an information need expressed as text

QREL
ground-truth relationship between a query and relevant passage

PAIR
(query, positive)

TRIPLET
(query, positive, negative)

SPLIT MANIFEST
article-level train/validation/test assignment
```

## Key methodological ideas

### Split before evaluation
All passages and queries from the same article stay in one split. This prevents
article-level leakage.

### Multiple relevant passages
A long article can produce several chunks, all of which remain relevant to an
article-level query.

### Separate semantic and reference queries
This prevents easy article-number matching from hiding weaknesses in semantic
retrieval.

### Deterministic synthetic queries
They provide a reproducible starting point, while their limitations remain explicit.

### Structural negatives are only the beginning
The current negatives are selected heuristically. A baseline embedding model can later
identify much more informative **hard negatives**.

---

## Next step — baseline retrieval evaluation

The next notebook should evaluate the original embedding model before any fine-tuning.

Conceptually:

```text
query
  ↓
baseline embedding model
  ↓
query vector

passages
  ↓
same embedding model
  ↓
passage vectors

query vector × passage vectors
  ↓
similarity ranking
  ↓
compare ranking with qrels
  ↓
Recall@K / MRR / nDCG
```

Recommended metrics:

- **Recall@1** — how much relevant content is found in the first result;
- **Recall@5** — how much is found in the first five;
- **Recall@10** — how much is found in the first ten;
- **MRR@10** — rewards placing the first relevant result near the top;
- **nDCG@10** — evaluates ranking quality while accounting for position.

Results should be reported separately for `reference` and `semantic` queries.

That baseline is essential: without measuring the original model first, we cannot later
claim that fine-tuning improved legal retrieval.